# 07 — Pemodelan prosedural 3D (pengganti CityEngine + CGA)

Aturan CGA diganti "rule file" Python (dict per kategori), dijalankan berurutan seperti CGA:

1. `extrude(footprint, H, z0)` → blok **LOD1**, digeser ke elevasi dasar.
2. `comp(block, footprint)` → atap (normal ke atas) dan fasad (daftar dinding: segmen + rentang z).
3. `split_floors(wall)` → lantai dasar `GFl`, lantai `Fl`, sisa parapet +1,0 m.
4. `split_tiles(wall, rule)` → tile dinding/jendela/pintu; pintu di lantai dasar fasad terpanjang;
   jendela = kotak tipis tertanam 0,15 m ke dinding (muka luar 2 cm di depan dinding agar terlihat) → **LOD2**.
5. `texture` → dinding diberi tekstur generik buatan PIL lewat UV (bukan foto berhak cipta).
6. Terrain DEM jadi mesh bertekstur citra; jalan dan air di-drape di atas terrain. Kampus memakai citra z19
   (0,3 m); cincin `context_m` di sekelilingnya memakai citra konteks z17 (~1,2 m) agar model tidak berakhir
   di tepi kotak AOI. Tekstur terrain disimpan JPEG di GLB (PNG ~5× lebih besar). Badan air di-*hydro-flatten*:
   muka air datar per danau, DEM di bawahnya diturunkan agar tepian DSM (kanopi) tidak menutupi air.
7. Ekspor `ui_lod1.glb`, `ui_lod1.obj`, `ui_lod2.glb` (Y-up sesuai glTF) dan catat waktu generate.

Koordinat model lokal: x/y relatif ke titik asal AOI, z relatif ke `z_ref`; disimpan di `output/origin.json`.

In [1]:
import io
import json
import time

import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import rasterio
import rasterio.features
import trimesh
from PIL import Image, ImageDraw
from pyproj import Transformer
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
from scipy.interpolate import RegularGridInterpolator
from shapely import affinity
from shapely.geometry.polygon import orient
from shapely.ops import unary_union
from trimesh.visual import TextureVisuals

from ui3d_common import CATEGORY_COLORS, CFG, CRS, aoi, fig_path, interim, out, raw, road_polygons

Bc = CFG["building"]
GFl, Fl, K = Bc["GFl"], Bc["Fl"], Bc["roof_const"]
gdb = out("ui3d.gpkg")
bld = gpd.read_file(gdb, layer="buildings").dropna(subset=["base_elev"]).reset_index(drop=True)
aoi_buf = aoi("aoi_buf50")

## Rule file (pengganti file CGA)

In [2]:
BASE_RULE = dict(tile_w=3.5, win_w=1.6, win_ratio=0.45, door_w=2.2, door_h=2.8,
                 window=[60, 80, 100], door=[90, 60, 40], roof=[120, 120, 120])
RULES = {
    "fakultas": dict(BASE_RULE, tile_w=3.2, win_w=1.8, win_ratio=0.5, roof=[150, 70, 50]),
    "rumah_sakit": dict(BASE_RULE, tile_w=3.0, win_w=1.4, roof=[200, 200, 200]),
    "masjid": dict(BASE_RULE, tile_w=4.0, win_w=1.2, win_ratio=0.6, door_w=3.0, door_h=3.2, roof=[30, 110, 70]),
    "olahraga": dict(BASE_RULE, tile_w=6.0, win_w=3.0, win_ratio=0.3, door_w=3.5, door_h=3.2, roof=[90, 100, 120]),
    "asrama": dict(BASE_RULE, tile_w=3.0, win_w=1.2, roof=[160, 80, 60]),
    "fasilitas_umum": dict(BASE_RULE),
}
INSET, OUT_OFFSET = 0.15, 0.02  # kedalaman jendela ke dalam dinding, tonjolan muka luar
TEX_M = 3.0                     # satu tekstur dinding mencakup 3 × 3 m

## Titik asal lokal dan helper mesh

In [3]:
c = aoi_buf.geometry.iloc[0].centroid
OX, OY = round(c.x), round(c.y)
with rasterio.open(out("dem_ui_8m.tif")) as s:
    win = rasterio.windows.from_bounds(*aoi_buf.total_bounds, s.transform).round_offsets().round_lengths()
    DEM = s.read(1, window=win)
    DEM_TR = s.window_transform(win)
DEM = np.where(np.isnan(DEM), np.nanmean(DEM), DEM)
Z_REF = float(np.floor(np.percentile(DEM, 1)))
lon, lat = Transformer.from_crs(CRS, "EPSG:4326", always_xy=True).transform(OX, OY)
json.dump(dict(x=OX, y=OY, z_ref=Z_REF, crs=CRS, lon=lon, lat=lat), open(out("origin.json"), "w"), indent=1)

ny, nx = DEM.shape
xs = DEM_TR.c + (np.arange(nx) + 0.5) * DEM_TR.a
ys = DEM_TR.f + (np.arange(ny) + 0.5) * DEM_TR.e
_interp = RegularGridInterpolator((ys[::-1], xs), DEM[::-1], bounds_error=False, fill_value=None)

# Grid DEM yang sama, diperlebar CTX m: offset piksel bulat → simpul kampus dan cincin berimpit (tanpa celah).
CTX = Bc["context_m"]
with rasterio.open(out("dem_ui_8m.tif")) as s:
    b = aoi_buf.total_bounds + np.r_[-CTX, -CTX, CTX, CTX]
    win = rasterio.windows.from_bounds(*b, s.transform).round_offsets().round_lengths()
    DEM_CTX = s.read(1, window=win)
    CTX_TR = s.window_transform(win)
DEM_CTX = np.where(np.isnan(DEM_CTX), np.nanmean(DEM_CTX), DEM_CTX)
cxs = CTX_TR.c + (np.arange(DEM_CTX.shape[1]) + 0.5) * CTX_TR.a
cys = CTX_TR.f + (np.arange(DEM_CTX.shape[0]) + 0.5) * CTX_TR.e


def dem_z(x, y):
    """Elevasi DEM (relatif Z_REF) di koordinat UTM."""
    return _interp(np.column_stack([y, x])) - Z_REF


class MeshBuilder:
    """Kumpulkan banyak potongan geometri jadi satu mesh (lebih cepat dari concatenate ribuan Trimesh)."""

    def __init__(self):
        self.v, self.f, self.c, self.uv, self.n = [], [], [], [], 0

    def add(self, v, f, color=None, uv=None):
        self.v.append(np.asarray(v, float))
        self.f.append(np.asarray(f) + self.n)
        self.n += len(v)
        if color is not None:
            self.c.append(np.tile(np.r_[color, 255], (len(f), 1)))
        if uv is not None:
            self.uv.append(np.asarray(uv, float))

    def mesh(self, image=None):
        if not self.v:
            return None
        m = trimesh.Trimesh(np.vstack(self.v), np.vstack(self.f), process=False)
        if image is not None:
            m.visual = TextureVisuals(uv=np.vstack(self.uv), image=image)
        elif self.c:
            m.visual.face_colors = np.vstack(self.c).astype(np.uint8)
        return m


_BOX = trimesh.creation.box(extents=(2, 2, 2))
EZ = np.array([0.0, 0.0, 1.0])


def oriented_box(center, u, n, half):
    """Kotak dengan sumbu (n, u, z) — basis tangan kanan agar winding template tetap benar."""
    V = _BOX.vertices
    return center + V[:, [0]] * half[0] * n + V[:, [1]] * half[1] * u + V[:, [2]] * half[2] * EZ


def local(poly):
    return orient(affinity.translate(poly, -OX, -OY), 1.0)

## Operasi prosedural: extrude → comp → split → texture

In [4]:
def extrude(poly, H, z0):
    m = trimesh.creation.extrude_polygon(poly, H)
    m.apply_translation([0, 0, z0])
    return m


def comp(block, poly):
    """Atap = face dengan normal ke atas; fasad = dinding per segmen ring (luar dan lubang)."""
    roof = block.submesh([np.where(block.face_normals[:, 2] > 0.9)[0]], append=True)
    z0, z1 = block.bounds[:, 2]
    walls = []
    for i, ring in enumerate([poly.exterior, *poly.interiors]):
        pts = np.asarray(ring.coords)
        for p0, p1 in zip(pts[:-1], pts[1:]):
            L = float(np.hypot(*(p1 - p0)))
            if L < 0.5:
                continue
            u = (p1 - p0) / L
            # ring luar CCW / lubang CW → sisi kanan arah jalan = luar bangunan
            walls.append(dict(p0=p0, L=L, u=np.r_[u, 0], n=np.r_[u[1], -u[0], 0], z0=z0, z1=z1, exterior=i == 0))
    return roof, walls


def split_floors(wall):
    """Pers. 1: lantai dasar GFl, N lantai Fl, sisa = parapet (+1,0 m)."""
    z0, z1 = wall["z0"], wall["z1"]
    bands, z = [(z0, z0 + GFl, "ground")], z0 + GFl
    while z + Fl <= z1 - K + 1e-6:
        bands.append((z, z + Fl, "floor"))
        z += Fl
    if z < z1 - 1e-6:
        bands.append((z, z1, "parapet"))
    return bands


def split_tiles(wall, rule):
    n = max(1, int(wall["L"] // rule["tile_w"]))
    e = np.linspace(0, wall["L"], n + 1)
    return list(zip(e[:-1], e[1:]))


def make_wall_texture(size=256):
    """Tekstur plester + garis panel, dibuat sendiri dengan PIL."""
    rng = np.random.default_rng(0)
    base = np.clip(rng.normal(225, 6, (size, size, 3)), 0, 255).astype(np.uint8)
    img = Image.fromarray(base)
    d = ImageDraw.Draw(img)
    for k in range(0, size, size // 4):
        d.line([(0, k), (size, k)], fill=(200, 196, 190), width=2)
    return img


WALL_TEX = make_wall_texture()


def texture(builder):
    return builder.mesh(image=WALL_TEX)

## Generate LOD1 dan LOD2

In [5]:
def generate(buildings, lod2=True):
    mb = {k: MeshBuilder() for k in ("lod1", "wall", "roof", "window", "door")}
    for _, r in buildings.iterrows():
        rule = RULES.get(r.category, BASE_RULE)
        polys = [r.geometry] if r.geometry.geom_type == "Polygon" else list(r.geometry.geoms)
        for p in polys:
            p = local(p)
            block = extrude(p, r.Bld_Height, r.base_elev - Z_REF)
            if not lod2:
                mb["lod1"].add(block.vertices, block.faces, color=CATEGORY_COLORS[r.category])
                continue
            roof, walls = comp(block, p)
            mb["roof"].add(roof.vertices, roof.faces, color=rule["roof"])
            ext = [w for w in walls if w["exterior"]]
            door_wall = max(ext, key=lambda w: w["L"]) if ext else None
            for w in walls:
                tiles = split_tiles(w, rule)
                for bz0, bz1, kind in split_floors(w):
                    for ti, (s0, s1) in enumerate(tiles):
                        a, b = w["p0"] + w["u"][:2] * s0, w["p0"] + w["u"][:2] * s1
                        V = [[*a, bz0], [*b, bz0], [*b, bz1], [*a, bz1]]
                        h0, h1 = bz0 - w["z0"], bz1 - w["z0"]
                        uv = [[s0 / TEX_M, h0 / TEX_M], [s1 / TEX_M, h0 / TEX_M], [s1 / TEX_M, h1 / TEX_M], [s0 / TEX_M, h1 / TEX_M]]
                        mb["wall"].add(V, [[0, 1, 2], [0, 2, 3]], uv=uv)
                        if kind == "parapet" or (s1 - s0) < rule["win_w"] + 0.4:
                            continue
                        mid = w["p0"] + w["u"][:2] * (s0 + s1) / 2
                        if kind == "ground" and w is door_wall and ti == len(tiles) // 2:
                            part, hw, hh, zc = "door", rule["door_w"] / 2, rule["door_h"] / 2, bz0 + rule["door_h"] / 2
                        else:
                            part, hw, hh = "window", rule["win_w"] / 2, (bz1 - bz0) * rule["win_ratio"] / 2
                            zc = (bz0 + bz1) / 2 + 0.2
                        center = np.r_[mid, zc] + w["n"] * (OUT_OFFSET - INSET / 2)
                        mb[part].add(oriented_box(center, w["u"], w["n"], (INSET / 2, hw, hh)), _BOX.faces,
                                     color=rule[part])
    if not lod2:
        return {"lod1_buildings": mb["lod1"].mesh()}
    return {"walls": texture(mb["wall"]), "roofs": mb["roof"].mesh(), "windows": mb["window"].mesh(),
            "doors": mb["door"].mesh()}


timing = {}
t0 = time.perf_counter(); lod1 = generate(bld, lod2=False); timing["LOD1"] = time.perf_counter() - t0
t0 = time.perf_counter(); lod2 = generate(bld, lod2=True); timing["LOD2"] = time.perf_counter() - t0
for k, v in timing.items():
    print(f"{k}: {v:.1f} s untuk {len(bld)} gedung ({v / len(bld) * 1000:.1f} ms/gedung)")

LOD1: 0.6 s untuk 995 gedung (0.6 ms/gedung)
LOD2: 3.5 s untuk 995 gedung (3.5 ms/gedung)


## Terrain, jalan, dan air (drape)

In [6]:
def jpeg(img, quality=90):
    """Array (3, h, w) → PIL JPEG; trimesh menyimpan JPEG apa adanya di GLB."""
    buf = io.BytesIO()
    Image.fromarray(img.transpose(1, 2, 0)).save(buf, format="JPEG", quality=quality)
    buf.seek(0)
    return Image.open(buf)


def grid_mesh(keep, tex_bounds, image):
    """Mesh dari grid DEM_CTX, hanya sel `keep`; UV memetakan tex_bounds (x0, y0, x1, y1) ke seluruh gambar."""
    X, Y = np.meshgrid(cxs, cys)
    V = np.column_stack([X.ravel() - OX, Y.ravel() - OY, DEM_CTX.ravel() - Z_REF])
    idx = np.arange(DEM_CTX.size).reshape(DEM_CTX.shape)
    a, b, c_, d = (q[keep] for q in (idx[:-1, :-1], idx[:-1, 1:], idx[1:, 1:], idx[1:, :-1]))
    F = np.vstack([np.column_stack([a, d, c_]), np.column_stack([a, c_, b])])
    x0, y0, x1, y1 = tex_bounds
    uv = np.column_stack([(X.ravel() - x0) / (x1 - x0), (Y.ravel() - y0) / (y1 - y0)])
    m = trimesh.Trimesh(V, F, process=False)
    m.visual = TextureVisuals(uv=uv, image=image)
    m.remove_unreferenced_vertices()
    return m


def terrain_mesh():
    """(kampus, cincin): grid yang sama, kampus bertekstur citra z19, cincin bertekstur citra konteks z17."""
    inner = (xs[0], ys[-1], xs[-1], ys[0])
    with rasterio.open(interim("ui_rgb.tif")) as s:
        w = rasterio.windows.from_bounds(*inner, s.transform)
        f = max(1, int(w.width // 4096) + 1)
        img = s.read(window=w, out_shape=(3, int(w.height // f), int(w.width // f)), boundless=True)
    ring = (cxs[0], cys[-1], cxs[-1], cys[0])
    with rasterio.open(raw("imagery_context")) as s, WarpedVRT(s, crs=CRS, resampling=Resampling.bilinear) as v:
        w = rasterio.windows.from_bounds(*ring, v.transform)
        k = min(1.0, 4096 / max(w.width, w.height))
        ctx = v.read((1, 2, 3), window=w, out_shape=(3, round(w.height * k), round(w.width * k)))
    mx, my = (cxs[:-1] + cxs[1:]) / 2, (cys[:-1] + cys[1:]) / 2  # pusat sel
    inside = ((my >= inner[1]) & (my <= inner[3]))[:, None] & ((mx >= inner[0]) & (mx <= inner[2]))[None, :]
    return grid_mesh(inside, inner, jpeg(img)), grid_mesh(~inside, ring, jpeg(ctx))


def drape(geoms, offset, color, levels=None):
    """Tempel poligon ke DEM. levels: satu elevasi datar per poligon (muka air), relatif Z_REF."""
    mb = MeshBuilder()
    for i, g in enumerate(geoms):
        if g.is_empty or g.geom_type != "Polygon":
            continue
        v2, f = trimesh.creation.triangulate_polygon(g.segmentize(4.0), engine="earcut")
        e1, e2 = v2[f[:, 1]] - v2[f[:, 0]], v2[f[:, 2]] - v2[f[:, 0]]
        f = np.where(((e1[:, 0] * e2[:, 1] - e1[:, 1] * e2[:, 0]) < 0)[:, None], f[:, ::-1], f)  # normal ke atas
        z = np.full(len(v2), levels[i]) if levels is not None else dem_z(v2[:, 0], v2[:, 1])
        mb.add(np.column_stack([v2[:, 0] - OX, v2[:, 1] - OY, z + offset]), f, color=color)
    return mb.mesh()


def burn_water(polys, levels, below=1.0):
    """Hydro-flattening: simpul DEM_CTX di dalam badan air diturunkan ke bawah muka air.

    DEM radar adalah DSM, jadi tepi danau ikut tinggi kanopi dan menutupi permukaan air. Setelah dibakar,
    muka air datar selalu terlihat; tepian naik dari dasar danau ke DSM di sekitarnya.
    """
    lv = rasterio.features.rasterize([(g, l + Z_REF - below) for g, l in zip(polys, levels)], out_shape=DEM_CTX.shape,
                                     transform=CTX_TR, fill=np.nan, dtype="float64")
    m = ~np.isnan(lv)
    DEM_CTX[m] = np.minimum(DEM_CTX[m], lv[m])
    return int(m.sum())


clip = aoi_buf.geometry.iloc[0]
roads = gpd.read_file(gdb, layer="roads")
water = gpd.read_file(gdb, layer="water")

# Muka air: satu level per badan air = persentil 10 DEM di dalamnya. Poligon air diperlebar WATER_PAD m agar
# celah antara tepi poligon dan simpul DEM 8 m yang diturunkan tertutup air (di darat, air tersembunyi di bawah DEM).
WATER_PAD = 12
lakes = [g for g in gpd.GeoSeries([unary_union(water.geometry).intersection(clip)]).explode(index_parts=False)
         if g.geom_type == "Polygon" and g.area > 50]
levels = [float(np.percentile(dem_z(*np.asarray(g.segmentize(4.0).exterior.coords).T), 10)) for g in lakes]
print(f"Badan air: {len(lakes)}; simpul DEM diturunkan: {burn_water(lakes, levels)}")

terrain, terrain_ctx = terrain_mesh()
context = {"terrain": terrain, "terrain_context": terrain_ctx,
           "roads": drape(gpd.GeoSeries([unary_union(road_polygons(roads)).intersection(clip)]).explode(index_parts=False),
                          0.5, [70, 70, 70]),
           "water": drape([g.buffer(WATER_PAD) for g in lakes], 0.0, [45, 95, 135], levels=levels)}

Badan air: 12; simpul DEM diturunkan: 3760


## Ekspor glb / obj (Z-up → Y-up glTF)

In [7]:
Y_UP = trimesh.transformations.rotation_matrix(-np.pi / 2, [1, 0, 0])


def export(parts, path):
    scene = trimesh.Scene({k: v for k, v in parts.items() if v is not None})
    scene.apply_transform(Y_UP)
    scene.export(path)
    print(path.name, f"{path.stat().st_size / 1e6:.1f} MB")
    return scene


export({**context, **lod1}, out("ui_lod1.glb"))
lod1["lod1_buildings"].copy().apply_transform(Y_UP).export(out("ui_lod1.obj"))
print("ui_lod1.obj ditulis")
export({**context, **lod2}, out("ui_lod2.glb"))

pd.DataFrame([dict(model=k, detik=round(v, 2), n_gedung=len(bld), ms_per_gedung=round(v / len(bld) * 1000, 2))
              for k, v in timing.items()]).to_csv(out("procedural_timing.csv"), index=False)

ui_lod1.glb 20.9 MB
ui_lod1.obj ditulis


ui_lod2.glb 39.7 MB


## Viewer (subset 150 m di sekitar gedung terbesar) dan figure (setara Fig. 11b, 12, 13)

In [8]:
cx, cy = bld.loc[bld.area.idxmax()].geometry.centroid.coords[0]
sub = bld[bld.centroid.distance(gpd.points_from_xy([cx], [cy])[0]) < 150]
sub_lod2 = generate(sub, lod2=True)
scene = trimesh.Scene({k: v for k, v in sub_lod2.items() if v is not None})
scene.show()


def plotly_mesh(parts, title, path):
    data = []
    for name, m in parts.items():
        if m is None:
            continue
        fc = m.visual.face_colors[:, :3] if m.visual.kind == "face" else np.tile([215, 210, 200], (len(m.faces), 1))
        data.append(go.Mesh3d(x=m.vertices[:, 0], y=m.vertices[:, 1], z=m.vertices[:, 2],
                              i=m.faces[:, 0], j=m.faces[:, 1], k=m.faces[:, 2],
                              facecolor=[f"rgb({r},{g},{b})" for r, g, b in fc], name=name, flatshading=True))
    f = go.Figure(data)
    f.update_layout(title=title, scene=dict(aspectmode="data"))
    f.write_html(fig_path(path))
    return f


plotly_mesh(generate(sub, lod2=False), "LOD1 — blok atap datar (setara Fig. 12)", "fig12_lod1.html").show()
plotly_mesh(sub_lod2, "LOD2 — fasad, jendela, pintu (setara Fig. 11b, 13)", "fig13_lod2.html").show()